# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, focusing on reproducible data access via the Croissant schema. All schema entities (record sets, fields, columns) are strictly referenced by their `@id` for clarity and FAIRness.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the mlcroissant Dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (single object - use attribute access, not dict subscripting)
print(f"Dataset: {dataset.metadata.name}\n")
print(f"Citation: {dataset.metadata.cite_as}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.date_published}\n")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references to the data entities should use their `@id` field, per Croissant specification.

In [ ]:
# List available record sets by their @id
print("Available record sets (referenced by @id):")
record_sets = dataset.metadata.record_sets
record_set_ids = []

for rs in record_sets:
    print(f" - @id: {rs.id}")
    print(f"   Name: {rs.name}")
    print(f"   Description: {getattr(rs, 'description', '')}")
    record_set_ids.append(rs.id)
    # List fields for each record set
    print("   Fields (by @id):")
    for field in rs.fields:
        print(f"     - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print()

# Print summary of record set IDs
print(f'All record set @ids: {record_set_ids}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Refer to record set and field `@id`s discovered in the previous section.

In [ ]:
# Example: Load all record sets into DataFrames using their @ids
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f'Loaded DataFrame for @id: {record_set_id} with {len(df)} rows, columns: {list(df.columns)}')
        else:
            print(f'No records found for @id: {record_set_id}')
    except Exception as e:
        print(f'Error loading records for {record_set_id}: {e}')

# Pick the main record set for further analysis (e.g., the first one)
main_record_set_id = record_set_ids[0]

print(f"\nColumns in main record set (@id: {main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All operations are performed referencing the relevant field/column `@id`.

In [ ]:
# For demo purposes, find the first numeric field for filtering, e.g., molecular_weight (use actual @id from prior step)
main_df = dataframes[main_record_set_id]
# Auto-select a numeric field (float/int columns) for demonstration
numeric_candidates = main_df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric field candidates: {numeric_candidates}")

if len(numeric_candidates) == 0:
    print("No numeric fields found in the main record set.")
else:
    numeric_field_id = numeric_candidates[0]

    # Filter records where numeric_field_id > threshold
    threshold = main_df[numeric_field_id].mean()  # Use mean as a meaningful threshold
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    # Normalize the selected numeric field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by a categorical field if available
    cat_fields = main_df.select_dtypes(include=['object']).columns.tolist()
    group_field_id = None
    # Find a groupable field (not too many unique values)
    for candidate in cat_fields:
        nunique = filtered_df[candidate].nunique()
        if 1 < nunique < 30:
            group_field_id = candidate
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. This example shows a distribution plot and scatter of the normalized value vs the original.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_candidates) > 0:
    # Distribution plot of normalized numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[col_norm], kde=True, bins=20)
    plt.title(f"Distribution of Normalized '{numeric_field_id}'")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.show()

    # Scatter plot: original vs normalized
    plt.figure(figsize=(8,6))
    sns.scatterplot(x=filtered_df[numeric_field_id], y=filtered_df[col_norm])
    plt.title(f"{numeric_field_id}: Raw vs Normalized")
    plt.xlabel(numeric_field_id)
    plt.ylabel(col_norm)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to access, explore, and analyze the FAIR² dataset using the `mlcroissant` library. By adhering to strict referencing via `@id` fields for all entities—including record sets, fields, and columns—we ensured full traceability and reproducibility. The analysis included metadata inspection, record set exploration, field-wise normalization, and basic visualization. You can extend this notebook to perform more advanced, domain-specific analyses using the rich data definitions of the Croissant schema.